In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
import warnings
warnings.filterwarnings('ignore')

In [ ]:
df = pd.read_csv('../data/cleaned_dataset.csv')
df['timestamp'] = pd.to_datetime(df['timestamp'])
print(f"Loaded: {df.shape}")
print(f"Columns: {list(df.columns)}")

In [ ]:
df['cpu_per_replica'] = df['cpu_usage'] / df['replica_count'].replace(0, 1)
df['memory_per_replica'] = df['memory_usage'] / df['replica_count'].replace(0, 1)
print("Created: cpu_per_replica, memory_per_replica")

In [ ]:
df['latency_diff'] = df['latency_p95'] - df['latency_p50']
print("Created: latency_diff")

In [ ]:
time_diff = df['timestamp'].diff().dt.total_seconds()
request_diff = df['request_rate'].diff()
df['traffic_change_rate'] = request_diff / time_diff.replace(0, 1)
print("Created: traffic_change_rate")

In [ ]:
for i in range(1, 6):
    df[f'request_rate_lag_{i}'] = df['request_rate'].shift(i)
    df[f'latency_p95_lag_{i}'] = df['latency_p95'].shift(i)
    df[f'cpu_usage_lag_{i}'] = df['cpu_usage'].shift(i)
print("Created: lag features (1-5 steps)")

In [ ]:
for window in [3, 5]:
    df[f'request_rate_rolling_mean_{window}'] = df['request_rate'].rolling(window=window).mean()
    df[f'latency_p95_rolling_mean_{window}'] = df['latency_p95'].rolling(window=window).mean()
    df[f'cpu_usage_rolling_mean_{window}'] = df['cpu_usage'].rolling(window=window).mean()
print("Created: rolling mean features (3, 5 window)")

In [ ]:
for window in [3, 5]:
    df[f'request_rate_rolling_std_{window}'] = df['request_rate'].rolling(window=window).std()
    df[f'latency_p95_rolling_std_{window}'] = df['latency_p95'].rolling(window=window).std()
    df[f'cpu_usage_rolling_std_{window}'] = df['cpu_usage'].rolling(window=window).std()
print("Created: rolling std features (3, 5 window)")

In [ ]:
print(f"\nBefore dropna: {len(df)}")
df_featured = df.dropna()
print(f"After dropna: {len(df_featured)}")
print(f"Total features: {df_featured.shape[1]}")

In [ ]:
print("\nFeature validation:")
print(f"Null values: {df_featured.isnull().sum().sum()}")
print(f"Infinite values: {np.isinf(df_featured.select_dtypes(include=[np.number])).sum().sum()}")
print(f"\nNew features statistics:")
print(df_featured[['cpu_per_replica', 'memory_per_replica', 'latency_diff', 'traffic_change_rate']].describe())

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

df_featured.plot(x='timestamp', y='cpu_per_replica', ax=axes[0,0], title='CPU per Replica', legend=False)
df_featured.plot(x='timestamp', y='memory_per_replica', ax=axes[0,1], title='Memory per Replica', legend=False, color='orange')
df_featured.plot(x='timestamp', y='latency_diff', ax=axes[1,0], title='Latency Diff (P95-P50)', legend=False, color='green')
df_featured.plot(x='timestamp', y='traffic_change_rate', ax=axes[1,1], title='Traffic Change Rate', legend=False, color='red')

plt.tight_layout()
plt.savefig('../results/img/feature_engineering_overview.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
df_featured.to_csv('../data/featured_dataset.csv', index=False)
print(f"Saved: featured_dataset.csv ({len(df_featured)} rows, {df_featured.shape[1]} features)")

In [ ]:
feature_metadata = {
    'original_rows': len(df),
    'featured_rows': len(df_featured),
    'original_features': 9,
    'total_features': df_featured.shape[1],
    'new_features': df_featured.shape[1] - 9,
    'feature_categories': {
        'per_replica': ['cpu_per_replica', 'memory_per_replica'],
        'derived': ['latency_diff', 'traffic_change_rate'],
        'lag_features': 15,
        'rolling_mean': 6,
        'rolling_std': 6
    },
    'rows_dropped': len(df) - len(df_featured),
    'retention_rate': len(df_featured) / len(df)
}

with open('feature_metadata.json', 'w') as f:
    json.dump(feature_metadata, f, indent=2)
    
print("\n=== FEATURE ENGINEERING SUMMARY ===")
print(f"Original features: {feature_metadata['original_features']}")
print(f"New features: {feature_metadata['new_features']}")
print(f"Total features: {feature_metadata['total_features']}")
print(f"Rows: {feature_metadata['featured_rows']}")
print(f"Retention: {feature_metadata['retention_rate']*100:.2f}%")
print("\n=== TASK 1.2 COMPLETE ===")